# Naive Bayes

In [26]:
#r "nuget: Sep, 0.12.1"

using nietras.SeparatedValues;

Installed Packages Sep, 0.12.1

In [27]:

record SpamDataSet(int Id, bool Spam, string Text, string[] Words);


IEnumerable<string> GetWords(string text) 
{
    return text.Split(" ").Where(x => x.All(char.IsAsciiLetter));
}

IEnumerable<SpamDataSet> ReadDatabase() {
    using var reader = Sep.Reader().FromFile("spam_ham_dataset.csv");

    foreach (var row in reader) {
        var text = row["text"].ToString();

        yield return new SpamDataSet(
            row[""].Parse<int>(),
            row["label"].ToString() == "spam",
            text,
            GetWords(text).ToArray()
        );
    }

}


var rawData = ReadDatabase().ToArray();
rawData.First().Words

[ enron, methanol, meter, is, a, follow, up, to, the, note, i, gave, you, on, monday, data, provided, by, daren, override ... (21 more) ]

In [28]:
var bagOfWords = rawData.SelectMany(x => x.Words).Distinct().ToArray();
bagOfWords

[ enron, methanol, meter, is, a, follow, up, to, the, note, i, gave, you, on, monday, data, provided, by, daren, override ... (40472 more) ]

In [29]:
static Random rng = new Random(420);

static void Shuffle<T>(this IList<T> list)  
{  
    int n = list.Count;  
    while (n > 1) {  
        n--;  
        int k = rng.Next(n + 1);  
        T value = list[k];  
        list[k] = list[n];  
        list[n] = value;  
    }  
}

rawData.Shuffle();
var splitIndex = (int)(rawData.Length * 0.8);

var train = rawData.Take(splitIndex).ToArray();
var test = rawData.Skip(splitIndex).ToArray();

(train.Length + test.Length, rawData.Length)

Item1,5171
Item2,5171


In [33]:
class WordCounter {

    public int Count {get; private set;} = 0;
    public int Spam {get; private set;} = 0;
    public int Ham => Count - Spam;

    public void AddCount(bool spam) {
        Count++;
        if (spam)
            Spam++;
    }

    public override string ToString() => $"({Count} / {Spam} / {Ham})";
}

Dictionary<string, WordCounter> counters = bagOfWords.ToDictionary(x => x, _ => new WordCounter());

foreach (var r in train) {
    foreach (var word in r.Words) counters[word].AddCount(r.Spam);
}

counters.Take(5)

index value 0 [enron, (4100 / 0 / 4100)] Key enron Value (4100 / 0 / 4100) Count 4100 Spam 0 Ham 4100 1 [methanol, (62 / 0 / 62)] Key methanol Value (62 / 0 / 62) Count 62 Spam 0 Ham 62 2 [meter, (1692 / 0 / 1692)] Key meter Value (1692 / 0 / 1692) Count 1692 Spam 0 Ham 1692 3 [is, (5217 / 1683 / 3534)] Key is Value (5217 / 1683 / 3534) Count 5217 Spam 1683 Ham 3534 4 [a, (7048 / 2665 / 4383)] Key a Value (7048 / 2665 / 4383) Count 7048 Spam 2665 Ham 4383

In [35]:
static double PSpam(this WordCounter wc) => (double)wc.Spam / (double)wc.Count;
static double PHam(this WordCounter wc) => (double)wc.Ham / (double)wc.Count;

static double PSpamL(this WordCounter wc, double m = 1) => ((double)wc.Spam + m * 1/(double)wc.Count) / ((double)wc.Count + m);
static double PHamL(this WordCounter wc, double m = 1) => ((double)wc.Ham + m * 1/(double)wc.Count) / ((double)wc.Count + m);

static double PSpam(this ICollection<SpamDataSet> ds) => (double)ds.Count(x => x.Spam) / (double)ds.Count;
static double PHam(this ICollection<SpamDataSet> ds) => (double)ds.Count(x => !x.Spam) / (double)ds.Count;

(train.PSpam(), train.PHam())

Item1,0.28820116054158607
Item2,0.7117988394584139


In [45]:
var PSpam = train.PSpam();
var PHam = train.PHam();

bool ClassifySpam(string text) {

    var pSpam = PSpam;
    var pHam = PHam;

    foreach (var word in GetWords(text)) {
        pSpam *= counters[word].PSpam();
        pHam *= counters[word].PHam();
    }

    return pSpam > pHam;
}

var check1 = test.Select(x => (x, ClassifySpam(x.Text))).ToArray();
var check2 = train.Select(x => (x, ClassifySpam(x.Text))).ToArray();

var correct = check1.Count(x => x.Item1.Spam == x.Item2);
var correct2 = check2.Count(x => x.Item1.Spam == x.Item2);

((double)correct / (double)check1.Length, (double)correct2 / (double)check2.Length)

Item1,0.7420289855072464
Item2,0.9799323017408124


In [41]:
var PSpam = train.PSpam();
var PHam = train.PHam();

bool ClassifySpamL(string text) {

    var pSpam = PSpam;
    var pHam = PHam;

    foreach (var word in GetWords(text)) {
        pSpam *= counters[word].PSpamL();
        pHam *= counters[word].PHamL();
    }

    return pSpam > pHam;
}

var check1 = test.Select(x => (x, ClassifySpamL(x.Text))).ToArray();

var correct = check1.Count(x => x.Item1.Spam == x.Item2);
(double)correct / (double)check1.Length

0.7381642512077294

In [42]:
var PSpam = train.PSpam();
var PHam = train.PHam();

bool ClassifySpamLog(string text) {

    var pSpam = Math.Log(PSpam);
    var pHam = Math.Log(PHam);

    foreach (var word in GetWords(text)) {
        pSpam += Math.Log(counters[word].PSpam());
        pHam += Math.Log(counters[word].PHam());
    }

    return pSpam > pHam;
}

var check1 = test.Select(x => (x, ClassifySpamLog(x.Text))).ToArray();

var correct = check1.Count(x => x.Item1.Spam == x.Item2);
(double)correct / (double)check1.Length

0.7420289855072464

In [44]:
var PSpam = train.PSpam();
var PHam = train.PHam();

bool ClassifySpamLogL(string text) {

    var pSpam = Math.Log(PSpam);
    var pHam = Math.Log(PHam);

    foreach (var word in GetWords(text)) {
        pSpam += Math.Log(counters[word].PSpamL());
        pHam += Math.Log(counters[word].PHamL());
    }

    return pSpam > pHam;
}

var check1 = test.Select(x => (x, ClassifySpamL(x.Text))).ToArray();

var correct = check1.Count(x => x.Item1.Spam == x.Item2);
(double)correct / (double)check1.Length

0.7381642512077294